## Linear Regression
Linear model:
$$y = Xw + \epsilon$$
> find weights $w$ that minimize squared error

$$min_w \Vert| Xw - y\Vert|^2$$


Solution:
$$w = (X^TX)^{-1} X^T y$$

## Ridge Regression
New objective:
$$min_w \Vert| Xw - y\Vert|^2 + \lambda \Vert| w\Vert|^2$$
Solution (ridge estimator):
$$w = (X^TX + \lambda I)^{-1} X^T y$$


## Bayesian Linear Regression
Assumes a Gaussian prior over weights and Gaussian observation noise. The posterior mean corresponds to a ridge regression, while the predictive variance naturally captures both observation noise and model uncertainty.

We are trying to learn:
$$y= w^Tx + \epsilon$$

> a distribution over possible weight vectors

Frequentist linear regression assumes there is one true parameter $w^*$, which is estimated from the data.
> Prior -> Likelihood -> Posterior -> Prediction

### Prior
Bayesian assumes weights are random variable instead of a fixed unknown number. It's what we believe before seeing data.

We put a prior distribution 
$$w \sim \mathcal{N}(0, \alpha^{-1}I)$$



### Likelihood
This is how data is generated given parameters. We assume noise:
$$y = Xw + \epsilon$$
$$\epsilon \sim \mathcal{N}(0, \beta^{-1}I)$$

so:
$$p(y|w) = \mathcal{N}(Xw, \beta^{-1}I)$$

### Posterior
After seeing data, we update belief about weights. We want $p(w|y)$
$$p(w|y) \propto p(y|w)p(w)$$

Since prior and likelihood are both Gaussian, posterior is also Gaussian:
$$p(w|y) = \mathcal{N}(m_N, S_N)$$

Posterior formulas:
$$S_N = (\alpha I + \beta X^TX)^{-1}$$

$$m_N = \beta S_N X^T y$$

Mathematically posterior mean is equivalent to ridge estimator. 
- prior precision $\alpha$ plays a role of $\lambda$

### Prediction
We want prediction for a new point $x_*$, by integrating over ALL possible weights:

$$p(y_* | x_*) = \int p(y_* | w, x_*) p (w | y)dw$$

Predictive mean:
$$\mu_* = x_*^T m_N$$

Predictive variance:
$$\sigma_*^2 = \beta^{-1} + x_*^TS_Nx_*$$

In [22]:
import torch

class BayesianLinearRegression:
    def __init__(self, alpha=1.0, beta=25.0):
        """
        alpha: prior precision on weights 
        beta: likelihood precision
        """
        self.alpha = alpha
        self.beta = beta

        self.m_N = None
        self.S_N = None 

    def fit(self, X, y):
        """
        X: (N, D)
        y: (N,)
        """
        N, D = X.shape
        I = torch.eye(D)
        
        self.S_N = torch.inverse(self.alpha * I + self.beta * X.T @ X)
        self.m_N = self.beta * self.S_N @ X.T @ y

    def predict(self, X):
        mean = (X @ self.m_N).squeeze(-1)

        tmp = (X @ self.S_N) * X
        quad = torch.sum((X @ self.S_N) * X, dim=1)

        var = 1.0/self.beta + quad
        std = torch.sqrt(var)
        return mean, std



In [23]:
torch.manual_seed(0)
x_raw = torch.linspace(0, 2, 30)
y = 2*x_raw + 1 + 0.2 * torch.randn(30)

X = torch.stack([torch.ones_like(x_raw), x_raw], dim=1)

model = BayesianLinearRegression(alpha=2.0, beta=25.0)
model.fit(X, y)

x_test_raw = torch.linspace(-0.5, 2.5, 100)
X_test = torch.stack([torch.ones_like(x_test_raw), x_test_raw], dim=1)

mean, std = model.predict(X_test)

print("Posterior mean:")
print(model.m_N.squeeze())

print("First 5 predictions:")
for i in range(5):
    print(
        f"x={x_test_raw[i].item():.2f}, "
        f"mean={mean[i].item():.3f}, "
        f"std={std[i].item():.3f}"
    )

Posterior mean:
tensor([0.9026, 2.0738])
First 5 predictions:
x=-0.50, mean=-0.134, std=0.223
x=-0.47, mean=-0.071, std=0.222
x=-0.44, mean=-0.009, std=0.221
x=-0.41, mean=0.054, std=0.221
x=-0.38, mean=0.117, std=0.220


## Variational Bayesian Regression

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class VariationalLinearRegression(nn.Module):
    def __init__(self, in_features):
        super().__init__()

        # variational posterior q(w) = N(mu, sigma^2)
        self.w_mu = nn.Parameter(torch.zeros(in_features, 1))
        self.w_rho = nn.Parameter(torch.zeros(in_features, 1))

        self.b_mu = nn.Parameter(torch.zeros(1))
        self.b_rho = nn.Parameter(torch.zeros(1))

    def sample_params(self):
        w_sigma = F.softplus(self.w_rho)
        b_sigma = F.softplus(self.b_rho)

        w_eps = torch.randn_like(self.w_mu)
        b_eps = torch.randn_like(self.b_mu)

        w = self.w_mu + w_sigma * w_eps
        b = self.b_mu + b_sigma * b_eps

        return w, b, w_sigma, b_sigma

    def forward(self, x):
        w, b, _, _ = self.sample_params()
        return x @ w + b

    def kl_divergence(self):
        """
        KL(q||p), assuming standard normal prior p(w)=N(0,1)
        """
        w_sigma = F.softplus(self.w_rho)
        b_sigma = F.softplus(self.b_rho)

        kl_w = 0.5 * torch.sum(
            self.w_mu**2 + w_sigma**2 - torch.log(w_sigma**2 + 1e-8) - 1
        )
        kl_b = 0.5 * torch.sum(
            self.b_mu**2 + b_sigma**2 - torch.log(b_sigma**2 + 1e-8) - 1
        )
        return kl_w + kl_b


# Example training
if __name__ == "__main__":
    torch.manual_seed(0)

    X = torch.linspace(0, 2, 50).unsqueeze(1)
    y = 2 * X + 1 + 0.2 * torch.randn_like(X)

    model = VariationalLinearRegression(in_features=1)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

    noise_var = 0.04  # assume known observation noise variance

    for epoch in range(2000):
        optimizer.zero_grad()

        y_pred = model(X)

        # negative log likelihood (up to constant)
        nll = torch.mean((y - y_pred) ** 2) / (2 * noise_var)

        kl = model.kl_divergence() / len(X)

        loss = nll + kl
        loss.backward()
        optimizer.step()

    print("Learned weight mean:", model.w_mu.item())
    print("Learned bias mean:", model.b_mu.item())